# M5 benchmark: агент с tool calling на той же LLM, что M3/M4

Этот notebook проверяет именно агентный режим M5, **не сравнение разных LLM**:

**текстовая цель → `google/gemma-4-26b-a4b-it` → последовательность tool calls → результат**

Фиксируем как в M3/M4: RouterAI, `temperature=0`, `D0=2026-09-01`, dataset `bank-tasks-v1` и те же 15 задач C1a…C3e. Меняется только исполнение: вместо одного `Query` модель сама выбирает атомарные инструменты.

Канонический набор M5: `search_tasks`, `filter_by_field`, `group_by`, `aggregate`, `find_extremum`, `sort`, `export_result`.

Для чистоты эксперимента здесь нет LangChain/PydanticAI: tool loop реализован напрямую через RouterAI. Запрос не использует `parallel_tool_calls`; все возвращённые tool calls исполняются и логируются последовательно. Скрытые рассуждения модели не запрашиваются и не сохраняются — только наблюдаемые LLM/tool события.

In [23]:
%pip -q install requests pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
MODEL = "google/gemma-4-26b-a4b-it"
ROUTERAI_BASE_URL = "https://routerai.ru/api/v1"
TEMPERATURE = 0.0
SEED = 0
TIMEOUT_SECONDS = 60
REFERENCE_DATE = "2026-09-01"
DATASET_SEED = 20260901
MAX_LLM_STEPS = 12
REPEATS = 5              # после smoke-test можно поставить 3-5
REQUIRE_FIRST_TOOL = True
print("Model:", MODEL)

Model: google/gemma-4-26b-a4b-it


In [25]:
import os
from dotenv import load_dotenv

load_dotenv()

ROUTERAI_API_KEY = os.getenv("ROUTERAI_API_KEY")
assert ROUTERAI_API_KEY and ROUTERAI_API_KEY.strip(), (
    "В .env не найден ROUTERAI_API_KEY"
)

print("ROUTERAI_API_KEY загружен из .env.")


ROUTERAI_API_KEY загружен из .env.


## 0. Smoke-test RouterAI без VPN

Запустите после загрузки `.env`. Если обе проверки дают HTTP 200, можно переходить к tool-calling benchmark.


In [26]:
import requests

print("=== 1. Проверка доступа к RouterAI ===")

headers = {
    "Authorization": f"Bearer {ROUTERAI_API_KEY}",
    "Content-Type": "application/json",
}

r = requests.get(
    ROUTERAI_BASE_URL.rstrip("/") + "/models",
    headers=headers,
    timeout=30,
)

print("GET /models:", r.status_code)

if r.status_code != 200:
    print(r.text[:1000])
    raise RuntimeError("RouterAI недоступен")

print("RouterAI доступен без VPN ✅")

print("\n=== 2. Проверка Gemma 4 ===")

r = requests.post(
    ROUTERAI_BASE_URL.rstrip("/") + "/chat/completions",
    headers=headers,
    json={
        "model": MODEL,
        "messages": [
            {"role": "user", "content": "Ответь только словом OK"}
        ],
        "temperature": 0,
        "max_tokens": 10,
    },
    timeout=60,
)

print("POST /chat/completions:", r.status_code)

if r.status_code != 200:
    print(r.text[:2000])
    raise RuntimeError("Запрос к Gemma завершился ошибкой")

data = r.json()

print("model:", data.get("model"))
print("answer:", data["choices"][0]["message"]["content"])
print("usage:", data.get("usage"))

print("\nGemma через RouterAI работает ✅")


=== 1. Проверка доступа к RouterAI ===
GET /models: 200
RouterAI доступен без VPN ✅

=== 2. Проверка Gemma 4 ===
POST /chat/completions: 200
model: google/gemma-4-26b-a4b-it
answer: OK
usage: {'total_tokens': 22, 'prompt_tokens': 20, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'completion_tokens': 2, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}, 'cost': 0.0001755676}

Gemma через RouterAI работает ✅


## 1. Детерминированный dataset и 15 экспериментальных задач

Notebook автономен: архив проекта загружать в Colab не требуется. Код ниже воспроизводит текущий генератор стенда: 120 записей, `seed=20260901`, `D0=2026-09-01`.

In [27]:
from datetime import date, timedelta
import random, json, time
from copy import deepcopy

STATUSES = ("Открыта", "В работе", "На ревью", "Готово", "Отменена")
PRIORITIES = ("Низкий", "Средний", "Высокий", "Критический")
EPICS = ("Авторизация", "Платежи", "Уведомления", "Профиль", "Инфраструктура")
LABELS = ("баг", "фича", "документация", "рефакторинг")
D0 = date.fromisoformat(REFERENCE_DATE)

def generate_dataset(seed=DATASET_SEED):
    rng = random.Random(seed)
    records=[]
    for index in range(120):
        records.append({
            "id": f"TASK-{index+1:03d}",
            "title": f"Задача банковского приложения № {index+1}",
            "status": rng.choice(STATUSES),
            "priority": rng.choice(PRIORITIES),
            "assignee": None if index % 9 == 0 else f"Участник {rng.randint(1,7)}",
            "epic": rng.choice(EPICS),
            "sprint": f"Спринт {rng.randint(1,10)}",
            "created_at": D0 - timedelta(days=round(index*89/119)),
            "deadline": None if index % 4 == 0 else D0 + timedelta(days=rng.randint(-30,30)),
            "labels": rng.sample(LABELS, rng.randint(1,2)),
            "estimate_hours": rng.randint(1,40),
        })
    anchors=[
        dict(status="В работе", priority="Высокий", epic="Платежи", deadline=D0-timedelta(days=1)),
        dict(status="На ревью", priority="Критический", labels=["баг"]),
        dict(status="Открыта", priority="Средний", deadline=None),
        dict(status="Готово", priority="Низкий"),
        dict(status="В работе", priority="Средний", epic="Авторизация", labels=["фича"], estimate_hours=12),
        dict(status="Открыта", priority="Высокий", sprint="Спринт 5", labels=["баг"], estimate_hours=16),
        dict(status="На ревью", priority="Средний", labels=["фича"]),
        dict(status="На ревью", priority="Средний", labels=["рефакторинг"]),
    ]
    for index, overrides in enumerate(anchors):
        records[index].update(created_at=D0-timedelta(days=index%5), assignee=f"Участник {index%7+1}")
        records[index].update(overrides)
    return records

RECORDS=generate_dataset()
assert len(RECORDS)==120
print("Dataset:", len(RECORDS), "records")

Dataset: 120 records


In [28]:
def F(field, operator, value=None): return {"field":field,"operator":operator,"value":value}
def recent(days): return [F("created_at","gte",(D0-timedelta(days=days)).isoformat()), F("created_at","lte",D0.isoformat())]
TASKS={}
def add(task_id, level, prompt, filters, aggregation=None, group=None, export=False):
    TASKS[task_id]={
        "task_id":task_id,"level":level,"tci":{1:3,2:9,3:13}[level],"prompt":prompt,
        "filters":filters,"aggregation":aggregation,
        "aggregation_field":"estimate_hours" if aggregation and aggregation!="COUNT" else None,
        "group":group,"requires_export":export,
    }

for suffix,status,priority in [("a","В работе","Высокий"),("b","На ревью","Критический"),("c","Открыта","Средний"),("d","Готово","Низкий"),("e","В работе","Средний")]:
    add(f"C1{suffix}",1,f"Показать задачи со статусом «{status}» и приоритетом «{priority}».",[F("status","eq",status),F("priority","eq",priority)])
add("C2a",2,"Задачи в эпике «Платежи», статус «В работе» ИЛИ «На ревью», за последние 14 дней, посчитать количество.",[F("epic","eq","Платежи"),F("status","in",["В работе","На ревью"]),*recent(14)],"COUNT")
add("C2b",2,"Задачи в спринте «Спринт 5», статус «Открыта» ИЛИ «В работе», приоритет «Высокий», посчитать суммарную оценку в часах.",[F("sprint","eq","Спринт 5"),F("status","in",["Открыта","В работе"]),F("priority","eq","Высокий")],"SUM")
add("C2c",2,"Задачи с меткой «баг», статус «В работе» ИЛИ «На ревью», за последние 30 дней, посчитать среднюю оценку.",[F("labels","eq","баг"),F("status","in",["В работе","На ревью"]),*recent(30)],"AVG")
add("C2d",2,"Задачи в эпике «Авторизация», статус «Открыта» ИЛИ «В работе», за последние 21 день, посчитать суммарную оценку в часах.",[F("epic","eq","Авторизация"),F("status","in",["Открыта","В работе"]),*recent(21)],"SUM")
add("C2e",2,"Задачи с меткой «фича», статус «На ревью» ИЛИ «Готово», за последние 7 дней, посчитать количество.",[F("labels","eq","фича"),F("status","in",["На ревью","Готово"]),*recent(7)],"COUNT")
add("C3a",3,"Исключить отменённые. Сгруппировать по эпикам, найти эпик с наибольшим числом просроченных (дедлайн прошёл, статус ≠ Готово), экспортировать.",[F("status","neq","Отменена"),F("deadline","lt",D0.isoformat()),F("status","neq","Готово")],"COUNT","epic",True)
add("C3b",3,"Исключить задачи без исполнителя. Сгруппировать по спринтам, найти спринт с наибольшей суммарной оценкой задач с меткой «баг» (оценка > 8 ч), экспортировать.",[F("assignee","not_null"),F("labels","eq","баг"),F("estimate_hours","gt",8)],"SUM","sprint",True)
add("C3c",3,"Исключить задачи с приоритетом «Низкий». Сгруппировать по исполнителям, найти исполнителя с наибольшим числом задач в статусе «Открыта» без дедлайна, экспортировать.",[F("priority","neq","Низкий"),F("status","eq","Открыта"),F("deadline","is_null"),F("assignee","not_null")],"COUNT","assignee",True)
add("C3d",3,"Исключить задачи с меткой «документация». Сгруппировать по эпикам, найти эпик с наибольшей суммарной оценкой задач в статусе «В работе» (оценка > 4 ч), экспортировать.",[F("labels","neq","документация"),F("status","eq","В работе"),F("estimate_hours","gt",4)],"SUM","epic",True)
add("C3e",3,"Исключить задачи, созданные более 60 дней назад. Сгруппировать по спринтам, найти спринт с наибольшим числом задач с меткой «рефакторинг» и статусом «На ревью», экспортировать.",[F("created_at","gte",(D0-timedelta(days=60)).isoformat()),F("labels","eq","рефакторинг"),F("status","eq","На ревью")],"COUNT","sprint",True)
assert len(TASKS)==15
print("Tasks:", ", ".join(TASKS))

Tasks: C1a, C1b, C1c, C1d, C1e, C2a, C2b, C2c, C2d, C2e, C3a, C3b, C3c, C3d, C3e


## 2. Локальная семантика атомарных инструментов

Инструменты не содержат LLM и не интерпретируют текст. Они только мутируют текущее состояние выборки. Это позволяет отделить ошибку планирования агента от ошибки executor-а.

In [29]:
def _coerce(field,value):
    if field in ("created_at","deadline") and isinstance(value,str): return date.fromisoformat(value)
    return value

def matches(record,field,operator,value=None):
    actual=record[field]
    if operator=="is_null": return actual is None
    if operator=="not_null": return actual is not None
    if actual is None: return False
    if operator in ("in","not_in"):
        vals=[_coerce(field,v) for v in value]
        ok=bool(set(actual).intersection(vals)) if field=="labels" else actual in vals
        return (not ok) if operator=="not_in" else ok
    expected=_coerce(field,value)
    if field=="labels":
        ok=expected in actual
        return (not ok) if operator=="neq" else ok
    return {"eq":actual==expected,"neq":actual!=expected,"gt":actual>expected,"gte":actual>=expected,"lt":actual<expected,"lte":actual<=expected}[operator]

def aggregate_value(records,operation,field=None):
    if operation=="COUNT": return len(records)
    vals=[r[field] for r in records]
    if not vals: return None
    if operation=="SUM": return sum(vals)
    if operation=="AVG": return sum(vals)/len(vals)
    if operation=="MAX": return max(vals)
    if operation=="MIN": return min(vals)
    raise ValueError(operation)

class ToolState:
    def __init__(self,records):
        self.all_records=deepcopy(records); self.current=deepcopy(records)
        self.group_field=None; self.group_values=None; self.last_aggregate=None
        self.selected_groups=[]; self.selected_records=None; self.extremum_value=None
        self.exported=False; self.export_scope=None
    def invalidate(self):
        self.group_field=None; self.group_values=None; self.last_aggregate=None
        self.selected_groups=[]; self.selected_records=None; self.extremum_value=None
        self.exported=False; self.export_scope=None
    def snapshot(self):
        return {"current_count":len(self.current),"group_field":self.group_field,"last_aggregate":self.last_aggregate,"selected_groups":self.selected_groups,"selected_count":None if self.selected_records is None else len(self.selected_records),"exported":self.exported}

def execute_tool(state,name,args):
    if name=="search_tasks":
        state.current=deepcopy(state.all_records); state.invalidate()
        return {"ok":True,"record_count":len(state.current),"fields":["id","title","status","priority","assignee","epic","sprint","created_at","deadline","labels","estimate_hours"]}
    if name=="filter_by_field":
        field=args["field"]; operator=args["operator"]; value=args.get("value")

        # Защита от типичной ошибки LLM:
        # operator="in"/"not_in" требует именно JSON-массив.
        # Невалидный вызов НЕ меняет текущее состояние, чтобы агент мог исправиться.
        if operator in ("in","not_in") and not isinstance(value,list):
            return {
                "ok":False,
                "error":"INVALID_ARGUMENT",
                "message":(
                    f"For operator '{operator}', value MUST be a JSON array, "
                    'for example ["Открыта", "В работе"]. '
                    f"Received {type(value).__name__}: {value!r}"
                ),
                "state_unchanged":True,
                "remaining_count":len(state.current),
            }

        # Для обычных сравнений массив, наоборот, считаем ошибкой.
        if operator not in ("in","not_in","is_null","not_null") and isinstance(value,list):
            return {
                "ok":False,
                "error":"INVALID_ARGUMENT",
                "message":f"For operator '{operator}', value MUST be a scalar, not an array.",
                "state_unchanged":True,
                "remaining_count":len(state.current),
            }

        state.current=[r for r in state.current if matches(r,field,operator,value)]
        state.invalidate()
        return {"ok":True,"remaining_count":len(state.current)}
    if name=="group_by":
        state.group_field=args["field"]; state.group_values=None; state.selected_groups=[]; state.selected_records=None; state.extremum_value=None
        keys=sorted({r[state.group_field] for r in state.current}, key=lambda x:(x is None,str(x)))
        return {"ok":True,"group_field":state.group_field,"group_count":len(keys),"groups":keys}
    if name=="aggregate":
        op=args["operation"]; field=args.get("field")
        if op!="COUNT" and not field: raise ValueError("SUM/AVG/MAX/MIN require field")
        if state.group_field:
            grouped={}
            for r in state.current: grouped.setdefault(r[state.group_field],[]).append(r)
            state.group_values={("__NULL__" if k is None else str(k)):aggregate_value(v,op,field) for k,v in grouped.items()}
            state.last_aggregate=None
            return {"ok":True,"operation":op,"group_values":state.group_values}
        state.last_aggregate=aggregate_value(state.current,op,field); state.group_values=None
        return {"ok":True,"operation":op,"value":state.last_aggregate}
    if name=="find_extremum":
        if not state.group_field or not state.group_values: raise ValueError("find_extremum requires group_by + aggregate")
        clean={k:v for k,v in state.group_values.items() if v is not None}
        if not clean:
            state.selected_groups=[]; state.selected_records=[]; state.extremum_value=None
            return {"ok":True,"selected_groups":[],"value":None,"record_count":0}
        best=(max if args["direction"]=="max" else min)(clean.values())
        state.selected_groups=[k for k,v in clean.items() if v==best]; state.extremum_value=best
        wire=lambda v:"__NULL__" if v is None else str(v)
        state.selected_records=[r for r in state.current if wire(r[state.group_field]) in state.selected_groups]
        return {"ok":True,"selected_groups":state.selected_groups,"value":best,"record_count":len(state.selected_records)}
    if name=="sort":
        field=args["field"]; reverse=args["direction"]=="desc"
        non=[r for r in state.current if r[field] is not None]; null=[r for r in state.current if r[field] is None]
        non.sort(key=lambda r:r[field],reverse=reverse); state.current=non+null
        return {"ok":True,"field":field,"direction":args["direction"],"record_count":len(state.current)}
    if name=="export_result":
        scope=args["scope"]
        if scope=="extremum_group":
            if state.selected_records is None: raise ValueError("extremum_group requires find_extremum")
            rows=state.selected_records
        else: rows=state.current
        state.exported=True; state.export_scope=scope
        return {"ok":True,"format":"csv","scope":scope,"record_count":len(rows),"record_ids":[r["id"] for r in rows],"aggregate":state.extremum_value if scope=="extremum_group" else state.last_aggregate,"selected_groups":state.selected_groups}
    raise KeyError(name)


### Self-test: защита `in/not_in`

Эта проверка должна пройти до запуска LLM:

- неправильная строка `"Открыта, В работе"` возвращает `INVALID_ARGUMENT`;
- состояние выборки при ошибке остаётся неизменным;
- правильный массив `["Открыта", "В работе"]` работает.


In [30]:

_guard_state = ToolState(RECORDS)

_bad = execute_tool(
    _guard_state,
    "filter_by_field",
    {"field":"status", "operator":"in", "value":"Открыта, В работе"},
)

assert _bad["ok"] is False
assert _bad["error"] == "INVALID_ARGUMENT"
assert _bad["state_unchanged"] is True
assert len(_guard_state.current) == 120

_good = execute_tool(
    _guard_state,
    "filter_by_field",
    {"field":"status", "operator":"in", "value":["Открыта", "В работе"]},
)

assert _good["ok"] is True
assert 0 < len(_guard_state.current) < 120

print("in/not_in guard: OK")
print("Невалидный вызов:", _bad)
print("Валидный вызов, remaining_count:", _good["remaining_count"])


in/not_in guard: OK
Невалидный вызов: {'ok': False, 'error': 'INVALID_ARGUMENT', 'message': 'For operator \'in\', value MUST be a JSON array, for example ["Открыта", "В работе"]. Received str: \'Открыта, В работе\'', 'state_unchanged': True, 'remaining_count': 120}
Валидный вызов, remaining_count: 39


In [31]:
def expected_for_task(task):
    rows=deepcopy(RECORDS)
    for f in task["filters"]: rows=[r for r in rows if matches(r,f["field"],f["operator"],f.get("value"))]
    if task["group"]:
        grouped={}
        for r in rows: grouped.setdefault(r[task["group"]],[]).append(r)
        vals={k:aggregate_value(v,task["aggregation"],task["aggregation_field"]) for k,v in grouped.items()}
        best=max(v for v in vals.values() if v is not None)
        selected=[k for k,v in vals.items() if v==best]
        rows=[r for r in rows if r[task["group"]] in selected]
        return {"record_ids":sorted(r["id"] for r in rows),"aggregate":best,"selected_groups":sorted("__NULL__" if x is None else str(x) for x in selected),"requires_export":True}
    val=aggregate_value(rows,task["aggregation"],task["aggregation_field"]) if task["aggregation"] else None
    return {"record_ids":sorted(r["id"] for r in rows),"aggregate":val,"selected_groups":[],"requires_export":False}

GROUND_TRUTH={k:expected_for_task(v) for k,v in TASKS.items()}

def evaluate_state(task_id,state):
    task=TASKS[task_id]; exp=GROUND_TRUTH[task_id]
    if task["group"]:
        rows=state.selected_records if state.selected_records is not None else []; agg=state.extremum_value; groups=sorted(state.selected_groups)
    else:
        rows=state.current; agg=state.last_aggregate if task["aggregation"] else None; groups=[]
    ids_ok=sorted(r["id"] for r in rows)==exp["record_ids"]
    aggregate_ok=(abs(float(agg)-float(exp["aggregate"]))<=0.01) if task["aggregation"]=="AVG" and agg is not None and exp["aggregate"] is not None else agg==exp["aggregate"]
    groups_ok=(groups==exp["selected_groups"]) if task["group"] else True
    export_ok=state.exported if exp["requires_export"] else True
    return {"correct":bool(ids_ok and aggregate_ok and groups_ok and export_ok),"ids_ok":ids_ok,"aggregate_ok":aggregate_ok,"groups_ok":groups_ok,"export_ok":export_ok,"expected_count":len(exp["record_ids"]),"actual_count":len(rows),"expected_aggregate":exp["aggregate"],"actual_aggregate":agg,"expected_groups":exp["selected_groups"],"actual_groups":groups}

## 3. Tool schemas, которые получает модель

Здесь намеренно нет универсального инструмента «выполни весь запрос»: иначе M5 превратится обратно в M3.

In [32]:
FIELD_ENUM=["id","title","status","priority","assignee","epic","sprint","created_at","deadline","labels","estimate_hours"]
GROUP_ENUM=["status","priority","assignee","epic","sprint"]
TOOLS=[
 {"type":"function","function":{"name":"search_tasks","description":"Начать работу с фиксированной базой или сбросить текущую выборку ко всем 120 записям.","parameters":{"type":"object","properties":{},"additionalProperties":False}}},
 {"type":"function","function":{"name":"filter_by_field","description":"Отфильтровать ТЕКУЩУЮ выборку по одному полю. Несколько условий выполняй последовательно. КРИТИЧЕСКИ ВАЖНО: для operator=in или operator=not_in поле value ОБЯЗАНО быть JSON-массивом, например [\\\"Открыта\\\", \\\"В работе\\\"], а не строкой \\\"Открыта, В работе\\\". Для labels eq=метка есть, neq=метки нет.","parameters":{"type":"object","properties":{"field":{"type":"string","enum":FIELD_ENUM},"operator":{"type":"string","enum":["eq","neq","gt","lt","gte","lte","in","not_in","is_null","not_null"]},"value":{"description":"Для in/not_in ОБЯЗАТЕЛЬНО JSON-массив значений; для is_null/not_null — null; для остальных операторов — одно скалярное значение.","anyOf":[{"type":"string"},{"type":"integer"},{"type":"array","items":{"anyOf":[{"type":"string"},{"type":"integer"}]}},{"type":"null"}]}},"required":["field","operator","value"],"additionalProperties":False}}},
 {"type":"function","function":{"name":"group_by","description":"Сгруппировать текущую выборку. После этого вызови aggregate.","parameters":{"type":"object","properties":{"field":{"type":"string","enum":GROUP_ENUM}},"required":["field"],"additionalProperties":False}}},
 {"type":"function","function":{"name":"aggregate","description":"Посчитать агрегат текущей выборки или каждой группы. COUNT: field=null; SUM/AVG/MAX/MIN: field=estimate_hours.","parameters":{"type":"object","properties":{"operation":{"type":"string","enum":["COUNT","SUM","AVG","MAX","MIN"]},"field":{"anyOf":[{"type":"string","enum":["estimate_hours"]},{"type":"null"}]}},"required":["operation","field"],"additionalProperties":False}}},
 {"type":"function","function":{"name":"find_extremum","description":"После group_by + aggregate выбрать группу/группы с максимальным или минимальным агрегатом.","parameters":{"type":"object","properties":{"direction":{"type":"string","enum":["max","min"]}},"required":["direction"],"additionalProperties":False}}},
 {"type":"function","function":{"name":"sort","description":"Сортировать текущую выборку только если пользователь явно просит сортировку.","parameters":{"type":"object","properties":{"field":{"type":"string","enum":FIELD_ENUM},"direction":{"type":"string","enum":["asc","desc"]}},"required":["field","direction"],"additionalProperties":False}}},
 {"type":"function","function":{"name":"export_result","description":"Экспортировать результат. После find_extremum используй scope=extremum_group. Вызывай только если пользователь явно просит экспорт.","parameters":{"type":"object","properties":{"format":{"type":"string","enum":["csv"]},"scope":{"type":"string","enum":["all_records","extremum_group"]}},"required":["format","scope"],"additionalProperties":False}}},
]
print([t["function"]["name"] for t in TOOLS])

['search_tasks', 'filter_by_field', 'group_by', 'aggregate', 'find_extremum', 'sort', 'export_result']


In [33]:
SYSTEM_PROMPT = 'Ты агент для работы с фиксированной базой задач банковского приложения.\nТвоя задача — выполнить пользовательскую цель ТОЛЬКО через доступные инструменты.\n\nДата эксперимента D0: {REFERENCE_DATE}. Все относительные даты вычисляй от D0, не от текущей даты.\n\nДопустимые значения:\n- status: Открыта, В работе, На ревью, Готово, Отменена\n- priority: Низкий, Средний, Высокий, Критический\n- epic: Авторизация, Платежи, Уведомления, Профиль, Инфраструктура\n- sprint: Спринт 1 ... Спринт 10\n- assignee: Участник 1 ... Участник 7; может отсутствовать\n- labels: баг, фича, документация, рефакторинг\n- created_at, deadline: YYYY-MM-DD; deadline может отсутствовать\n- estimate_hours: целое число часов\n\nПравила:\n1. Не придумывай данные и не отвечай из общих знаний — используй инструменты.\n2. Независимые фильтры применяй последовательными вызовами filter_by_field.\n3. Альтернативы одного поля через «ИЛИ» передавай одним filter_by_field с operator=in. При operator=in/not_in value ОБЯЗАТЕЛЬНО передавай как JSON-массив, например ["Открыта", "В работе"]; никогда не передавай строку "Открыта, В работе".\n4. «За последние N дней»: created_at >= D0-N дней и created_at <= D0, обе границы включены.\n5. «Исключить созданные более N дней назад»: created_at >= D0-N дней.\n6. «Просроченные»: deadline < D0 И status != "Готово".\n7. «Без дедлайна»: deadline is_null.\n8. «Исключить задачи без исполнителя»: assignee not_null.\n9. Для labels: eq = метка присутствует, neq = метка отсутствует.\n10. Для группового экстремума порядок обязателен: group_by → aggregate → find_extremum.\n11. Если пользователь просит экспорт, после вычисления результата обязательно вызови export_result.\n12. Не вызывай sort, если сортировка явно не нужна.\n13. Когда задача полностью выполнена, дай очень короткое подтверждение.'.format(REFERENCE_DATE=REFERENCE_DATE)
print(SYSTEM_PROMPT[:600], '...')

Ты агент для работы с фиксированной базой задач банковского приложения.
Твоя задача — выполнить пользовательскую цель ТОЛЬКО через доступные инструменты.

Дата эксперимента D0: 2026-09-01. Все относительные даты вычисляй от D0, не от текущей даты.

Допустимые значения:
- status: Открыта, В работе, На ревью, Готово, Отменена
- priority: Низкий, Средний, Высокий, Критический
- epic: Авторизация, Платежи, Уведомления, Профиль, Инфраструктура
- sprint: Спринт 1 ... Спринт 10
- assignee: Участник 1 ... Участник 7; может отсутствовать
- labels: баг, фича, документация, рефакторинг
- created_at, dead ...


## 4. Самопроверка инструментов без LLM

Перед benchmark эта ячейка должна дать **15/15**. Она доказывает, что все задачи решаются выбранными семью атомарными инструментами и ground truth согласован с executor-ом.

In [34]:
def run_canonical_plan(task_id):
    task=TASKS[task_id]; state=ToolState(RECORDS)
    execute_tool(state,"search_tasks",{})
    for f in task["filters"]: execute_tool(state,"filter_by_field",f)
    if task["group"]: execute_tool(state,"group_by",{"field":task["group"]})
    if task["aggregation"]: execute_tool(state,"aggregate",{"operation":task["aggregation"],"field":task["aggregation_field"]})
    if task["group"]: execute_tool(state,"find_extremum",{"direction":"max"})
    if task["requires_export"]: execute_tool(state,"export_result",{"format":"csv","scope":"extremum_group"})
    return evaluate_state(task_id,state)
checks={k:run_canonical_plan(k) for k in TASKS}
bad={k:v for k,v in checks.items() if not v["correct"]}
assert not bad,bad
print(f"Tool self-test: {len(checks)}/{len(checks)} correct")

Tool self-test: 15/15 correct


## 5. Прямой RouterAI agent loop

Метрики на каждую попытку: корректность, LLM calls, tool calls, сумма LLM latency, локальное tool time, total latency, input/output tokens, provider/model и полная observable-траектория.

## RouterAI + Gemma 4: параметры agent loop

Для M5 отправляется минимальный OpenAI-compatible набор параметров:

- `model`
- `temperature`
- `messages`
- `tools`
- `tool_choice="auto"`

На этом этапе не добавляем gateway-specific routing-параметры: сначала проверяем поведение той же Gemma 4 через RouterAI без изменения логики M5.

Модель остаётся `google/gemma-4-26b-a4b-it`, `temperature=0`. Фактический `response_model`, токены и latency логируются для каждой попытки.


In [35]:
import requests

def selected_provider(payload):
    """
    RouterAI may or may not expose the underlying inference provider
    in the OpenAI-compatible response. Log it when present; otherwise None.
    """
    if not isinstance(payload, dict):
        return None
    return payload.get("provider") or payload.get("provider_name")

def call_routerai(messages, first_turn=False):
    """
    Minimal OpenAI-compatible tool-calling request for Gemma 4 via RouterAI.
    The agent policy/tool schemas are unchanged; only the gateway changed.
    """
    body = {
        "model": MODEL,
        "temperature": TEMPERATURE,
        "messages": messages,
        "tools": TOOLS,
        "tool_choice": "auto",
    }

    t0 = time.perf_counter()
    r = requests.post(
        ROUTERAI_BASE_URL.rstrip("/") + "/chat/completions",
        headers={
            "Authorization": f"Bearer {ROUTERAI_API_KEY}",
            "Content-Type": "application/json",
        },
        json=body,
        timeout=TIMEOUT_SECONDS,
    )
    ms = (time.perf_counter() - t0) * 1000

    if not r.ok:
        raise RuntimeError(f"RouterAI HTTP {r.status_code}: {r.text[:2000]}")

    p = r.json()
    msg = p["choices"][0]["message"]
    usage = p.get("usage") or {}

    return {
        "message": msg,
        "elapsed_ms": ms,
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "provider": selected_provider(p),
        "response_model": p.get("model"),
        "raw": p,
    }


In [36]:
def run_agent(task_id,run_index=1):
    task=TASKS[task_id]; state=ToolState(RECORDS)
    messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":task["prompt"]}]
    trajectory=[]; llm_ms=tool_ms=0.0; llm_calls=tool_calls_n=prompt_tokens=completion_tokens=0
    providers=[]; response_models=[]; termination=error=final_text=None
    t_total=time.perf_counter()
    try:
        for step in range(1,MAX_LLM_STEPS+1):
            resp=call_routerai(messages,first_turn=(step==1)); llm_calls+=1; llm_ms+=resp["elapsed_ms"]
            prompt_tokens+=resp["prompt_tokens"] or 0; completion_tokens+=resp["completion_tokens"] or 0
            if resp["provider"]: providers.append(resp["provider"])
            if resp["response_model"]: response_models.append(resp["response_model"])
            msg=resp["message"]; calls=msg.get("tool_calls") or []; final_text=msg.get("content")
            trajectory.append({"step":step,"type":"llm","latency_ms":round(resp["elapsed_ms"],2),"content":final_text,"tool_calls_requested":len(calls),"provider":resp["provider"],"response_model":resp["response_model"],"prompt_tokens":resp["prompt_tokens"],"completion_tokens":resp["completion_tokens"]})
            assistant={"role":"assistant","content":msg.get("content")}
            if calls: assistant["tool_calls"]=calls
            messages.append(assistant)
            if not calls: termination="model_finished"; break
            for tc in calls:
                name=tc["function"]["name"]
                try: args=json.loads(tc["function"].get("arguments") or "{}")
                except json.JSONDecodeError: args={"__invalid_json__":tc["function"].get("arguments")}
                t_tool=time.perf_counter()
                try:
                    if "__invalid_json__" in args: raise ValueError("Invalid JSON arguments")
                    result=execute_tool(state,name,args); ok=True
                except Exception as exc:
                    result={"ok":False,"error":type(exc).__name__,"message":str(exc)}; ok=False
                one_ms=(time.perf_counter()-t_tool)*1000; tool_ms+=one_ms; tool_calls_n+=1
                trajectory.append({"step":step,"type":"tool","tool":name,"args":args,"result":result,"ok":ok,"latency_ms":round(one_ms,4),"state":state.snapshot()})
                messages.append({"role":"tool","tool_call_id":tc["id"],"content":json.dumps(result,ensure_ascii=False,default=str)})
        else: termination="max_steps"
    except Exception as exc:
        termination="exception"; error=f"{type(exc).__name__}: {exc}"
    total_ms=(time.perf_counter()-t_total)*1000; ev=evaluate_state(task_id,state)
    return {"task_id":task_id,"level":task["level"],"tci":task["tci"],"run":run_index,"correct":ev["correct"],"ids_ok":ev["ids_ok"],"aggregate_ok":ev["aggregate_ok"],"groups_ok":ev["groups_ok"],"export_ok":ev["export_ok"],"llm_calls":llm_calls,"tool_calls":tool_calls_n,"llm_ms":round(llm_ms,2),"tool_ms":round(tool_ms,4),"total_ms":round(total_ms,2),"prompt_tokens":prompt_tokens,"completion_tokens":completion_tokens,"termination":termination,"error":error,"provider":providers[-1] if providers else None,"providers_seen":providers,"response_model":response_models[-1] if response_models else None,"final_text":final_text,"expected_count":ev["expected_count"],"actual_count":ev["actual_count"],"expected_aggregate":ev["expected_aggregate"],"actual_aggregate":ev["actual_aggregate"],"expected_groups":ev["expected_groups"],"actual_groups":ev["actual_groups"],"trajectory":trajectory}

## 6. Smoke-test: C2a

Сначала запустите одну задачу. Если `Correct: True`, переходите к полному benchmark.

In [37]:
SMOKE_TASK_ID="C2a"
smoke=run_agent(SMOKE_TASK_ID)
print("Prompt:",TASKS[SMOKE_TASK_ID]["prompt"])
for k in ["correct","llm_calls","tool_calls","llm_ms","total_ms","provider","termination","error"]: print(f"{k}: {smoke[k]}")

Prompt: Задачи в эпике «Платежи», статус «В работе» ИЛИ «На ревью», за последние 14 дней, посчитать количество.
correct: True
llm_calls: 6
tool_calls: 5
llm_ms: 9522.8
total_ms: 9547.75
provider: NextBit
termination: model_finished
error: None


In [38]:
import pandas as pd

def trajectory_frame(run):
    rows=[]
    for e in run["trajectory"]:
        if e["type"]=="llm": rows.append({"step":e["step"],"type":"LLM","tool":None,"args":None,"result":e.get("content"),"latency_ms":e["latency_ms"]})
        else: rows.append({"step":e["step"],"type":"TOOL","tool":e["tool"],"args":json.dumps(e["args"],ensure_ascii=False),"result":json.dumps(e["result"],ensure_ascii=False,default=str),"latency_ms":e["latency_ms"]})
    return pd.DataFrame(rows)
display(trajectory_frame(smoke))

,step,type,tool,args,result,latency_ms
0,1,LLM,NaN,NaN,NaN,1936.0200
1,1,TOOL,search_tasks,{},"{""ok"": true, ""record_count"": 120, ""fields"": [""...",4.9996
2,2,LLM,NaN,NaN,NaN,1193.9100
3,2,TOOL,filter_by_field,"{""field"": ""epic"", ""operator"": ""eq"", ""value"": ""...","{""ok"": true, ""remaining_count"": 29}",0.2652
4,3,LLM,NaN,NaN,NaN,1942.2700
5,3,TOOL,filter_by_field,"{""field"": ""status"", ""operator"": ""in"", ""value"":...","{""ok"": true, ""remaining_count"": 10}",0.1028
6,4,LLM,NaN,NaN,NaN,2151.7500
7,4,TOOL,filter_by_field,"{""field"": ""created_at"", ""operator"": ""gte"", ""va...","{""ok"": true, ""remaining_count"": 1}",0.0665
8,5,LLM,NaN,NaN,NaN,1009.2700
9,5,TOOL,aggregate,"{""field"": null, ""operation"": ""COUNT""}","{""ok"": true, ""operation"": ""COUNT"", ""value"": 1}",0.0129



### Что проверяем этим прогоном

`MAX_LLM_STEPS` намеренно остаётся равным **12**.  
Если после защиты аргументов C2b–C2e перестанут зацикливаться и завершатся до лимита, значит проблема была именно в контракте `filter_by_field`, а не в недостаточном количестве агентских шагов.


## 7. Полный benchmark: 15 задач × REPEATS

На первом запуске оставьте `REPEATS=1`. Для оценки стабильности увеличьте до `3` или `5`.

In [39]:
all_runs=[]
for rep in range(1,REPEATS+1):
    for task_id in TASKS:
        print(f"[{rep}/{REPEATS}] {task_id} ...",end=" ")
        run=run_agent(task_id,rep); all_runs.append(run)
        print("OK" if run["correct"] else "FAIL",f"| tools={run['tool_calls']} | llm={run['llm_calls']} | {run['total_ms']:.0f} ms")
print("Done:",len(all_runs),"runs")

[1/5] C1a ... 

OK | tools=3 | llm=4 | 6821 ms
[1/5] C1b ... OK | tools=3 | llm=3 | 63489 ms
[1/5] C1c ... OK | tools=3 | llm=4 | 15474 ms
[1/5] C1d ... OK | tools=3 | llm=4 | 12266 ms
[1/5] C1e ... OK | tools=3 | llm=4 | 6571 ms
[1/5] C2a ... OK | tools=5 | llm=6 | 9214 ms
[1/5] C2b ... OK | tools=5 | llm=6 | 8674 ms
[1/5] C2c ... OK | tools=5 | llm=6 | 8693 ms
[1/5] C2d ... OK | tools=5 | llm=6 | 10498 ms
[1/5] C2e ... OK | tools=5 | llm=6 | 8289 ms
[1/5] C3a ... OK | tools=8 | llm=9 | 12592 ms
[1/5] C3b ... OK | tools=8 | llm=9 | 12210 ms
[1/5] C3c ... OK | tools=8 | llm=9 | 10457 ms
[1/5] C3d ... OK | tools=8 | llm=9 | 12920 ms
[1/5] C3e ... OK | tools=8 | llm=9 | 12897 ms
[2/5] C1a ... OK | tools=3 | llm=4 | 8222 ms
[2/5] C1b ... OK | tools=3 | llm=4 | 5884 ms
[2/5] C1c ... OK | tools=3 | llm=4 | 14715 ms
[2/5] C1d ... OK | tools=3 | llm=4 | 13345 ms
[2/5] C1e ... OK | tools=3 | llm=4 | 7126 ms
[2/5] C2a ... OK | tools=5 | llm=6 | 8857 ms
[2/5] C2b ... OK | tools=5 | llm=6 | 7746 ms
[2/5] C2c ...

In [40]:
RESULT_COLUMNS=["task_id","level","tci","run","correct","ids_ok","aggregate_ok","groups_ok","export_ok","llm_calls","tool_calls","llm_ms","tool_ms","total_ms","prompt_tokens","completion_tokens","termination","provider","response_model","error"]
results_df=pd.DataFrame([{k:r.get(k) for k in RESULT_COLUMNS} for r in all_runs])
display(results_df)

,task_id,level,tci,run,correct,ids_ok,aggregate_ok,groups_ok,export_ok,llm_calls,tool_calls,llm_ms,tool_ms,total_ms,prompt_tokens,completion_tokens,termination,provider,response_model,error
0,C1a,1,3,1,True,True,True,True,True,4,3,6806.54,4.5721,6821.48,6031,188,model_finished,NextBit,google/gemma-4-26b-a4b-it,NaN
1,C1b,1,3,1,True,True,True,True,True,3,3,3161.09,5.2377,63489.41,4434,80,exception,NextBit,google/gemma-4-26b-a4b-it,ConnectionError: HTTPSConnectionPool(host='rou...
2,C1c,1,3,1,True,True,True,True,True,4,3,15456.44,4.9410,15473.58,6033,1066,model_finished,NextBit,google/gemma-4-26b-a4b-it,NaN
3,C1d,1,3,1,True,True,True,True,True,4,3,12252.34,4.5185,12266.16,6031,763,model_finished,NextBit,google/gemma-4-26b-a4b-it,NaN
4,C1e,1,3,1,True,True,True,True,True,4,3,6553.53,6.1487,6570.94,6026,282,model_finished,NextBit,google/gemma-4-26b-a4b-it,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,C3a,3,13,5,True,True,True,True,True,9,8,8248.56,10.3840,8290.39,15438,236,model_finished,NextBit,google/gemma-4-26b-a4b-it,NaN
71,C3b,3,13,5,True,True,True,True,True,9,8,7287.98,9.7610,7332.17,15725,232,model_finished,NextBit,google/gemma-4-26b-a4b-it,NaN
72,C3c,3,13,5,True,True,True,True,True,9,8,6861.21,4.9934,6893.58,15476,242,model_finished,NextBit,google/gemma-4-26b-a4b-it,NaN
73,C3d,3,13,5,True,True,True,True,True,9,8,7049.07,4.9199,7086.76,15403,238,model_finished,NextBit,google/gemma-4-26b-a4b-it,NaN


In [41]:
summary_by_level=(results_df.groupby(["level","tci"],as_index=False).agg(
    runs=("correct","size"),accuracy_pct=("correct",lambda s:round(100*s.mean(),2)),median_total_ms=("total_ms","median"),median_llm_ms=("llm_ms","median"),mean_llm_calls=("llm_calls","mean"),mean_tool_calls=("tool_calls","mean"),mean_prompt_tokens=("prompt_tokens","mean"),mean_completion_tokens=("completion_tokens","mean")))
display(summary_by_level)
print("Overall accuracy, %:",round(100*results_df["correct"].mean(),2))

,level,tci,runs,accuracy_pct,median_total_ms,median_llm_ms,mean_llm_calls,mean_tool_calls,mean_prompt_tokens,mean_completion_tokens
0,1,3,25,100.0,8222.27,7856.76,3.96,3.00,5967.48,437.12
1,2,9,25,92.0,6867.25,6784.45,5.60,4.68,8899.08,180.56
2,3,13,25,100.0,8290.39,8235.33,9.00,8.00,15514.80,241.20


Overall accuracy, %: 97.33


In [42]:
failed=[r for r in all_runs if not r["correct"]]
print("Failed runs:",len(failed))
for run in failed[:5]:
    print("\n"+"="*80); print(run["task_id"],TASKS[run["task_id"]]["prompt"])
    print({k:run[k] for k in ["ids_ok","aggregate_ok","groups_ok","export_ok","termination","error"]})
    display(trajectory_frame(run))

Failed runs: 2

C2d Задачи в эпике «Авторизация», статус «Открыта» ИЛИ «В работе», за последние 21 день, посчитать суммарную оценку в часах.
{'ids_ok': False, 'aggregate_ok': False, 'groups_ok': True, 'export_ok': True, 'termination': 'exception', 'error': 'ChunkedEncodingError: Response ended prematurely'}


,step,type,tool,args,result,latency_ms
0,1,LLM,NaN,NaN,NaN,988.5500
1,1,TOOL,search_tasks,{},"{""ok"": true, ""record_count"": 120, ""fields"": [""...",5.4699
2,2,LLM,NaN,NaN,NaN,893.0100
3,2,TOOL,filter_by_field,"{""field"": ""epic"", ""operator"": ""eq"", ""value"": ""...","{""ok"": true, ""remaining_count"": 20}",0.3260



C2e Задачи с меткой «фича», статус «На ревью» ИЛИ «Готово», за последние 7 дней, посчитать количество.
{'ids_ok': False, 'aggregate_ok': False, 'groups_ok': True, 'export_ok': True, 'termination': 'exception', 'error': "SSLError: HTTPSConnectionPool(host='routerai.ru', port=443): Max retries exceeded with url: /api/v1/chat/completions (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1032)')))"}


""


## 8. Сохранить результаты

CSV — сводные метрики; JSON — полные наблюдаемые траектории агента.

In [ ]:
results_df.to_csv("m5_benchmark_results.csv",index=False,encoding="utf-8-sig")
with open("m5_benchmark_trajectories.json","w",encoding="utf-8") as f: json.dump(all_runs,f,ensure_ascii=False,indent=2,default=str)
print("Saved: m5_benchmark_results.csv, m5_benchmark_trajectories.json")

Saved: m5_benchmark_results.csv, m5_benchmark_trajectories.json


: 

# Что смотреть в результате

Основные показатели M5: **task completion / accuracy**, accuracy отдельно по C1/C2/C3, число `tool_calls`, число `llm_calls`, `llm_ms`, `total_ms`, токены и типы ошибок. Для неудачных задач смотрите trajectory: она показывает, какой инструмент и с какими аргументами выбрала модель.

Для стенда потом переносятся те же семь схем инструментов, capped loop, последовательное исполнение и журнал каждого LLM/tool события и существующая ground-truth проверка. Модель между M3, M4 и M5 не меняется.